In [ ]:
##############################################################################
# RESUME SCRIPT — run top to bottom in a fresh Kaggle kernel.
# Datasets needed: kan-qwen, llama-kan, images-new-known,
#                  kan_route_TRULY_FINAL_v2 (zip), AND your KAN_ROUTE_PART4_FINAL
#                  zip re-uploaded as a dataset (to salvage 12 checkpoints).
##############################################################################
!pip install -q open_clip_torch
# --------------------------------------------------------------------------
# Cell 0 — Setup, load everything
# --------------------------------------------------------------------------
from pathlib import Path
import json, os, sys, re, shutil
import numpy as np, torch, torch.nn as nn, pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
print(f"Device: {DEVICE}")

def find_dir(*fragments):
    for frag in fragments:
        m = [p for p in Path("/kaggle/input").rglob("*") if p.is_dir() and frag in p.name.lower()]
        if m: return m[0]
    return None

KAN_QWEN = find_dir("kan_qwen", "kan-qwen")
KAN_LLAMA = find_dir("kan_llama", "llama-kan")
KNOWN_IMG_DIR = find_dir("known")
NEW_IMG_DIR = find_dir("new")
for name, p in [("KAN_QWEN", KAN_QWEN), ("KAN_LLAMA", KAN_LLAMA),
                ("KNOWN_IMG_DIR", KNOWN_IMG_DIR), ("NEW_IMG_DIR", NEW_IMG_DIR)]:
    print(f"{'✅' if p else '❌'} {name}: {p}")

_human_csv_matches = list(Path("/kaggle/input").rglob("human_queries_cleaned_TRUE_FINAL.csv"))
assert _human_csv_matches, "kan_route_TRULY_FINAL_v2 zip not attached."
human_df = pd.read_csv(_human_csv_matches[0])
stale_cols = [c for c in human_df.columns if c.endswith("_x") or c.endswith("_y")]
human_df = human_df.drop(columns=stale_cols) if stale_cols else human_df
print(f"✅ Human benchmark: {len(human_df)} rows")

if KNOWN_IMG_DIR == NEW_IMG_DIR:
    subdirs = {p.name.lower(): p for p in KNOWN_IMG_DIR.iterdir() if p.is_dir()}
    if "known" in subdirs and "new" in subdirs:
        KNOWN_IMG_DIR, NEW_IMG_DIR = subdirs["known"], subdirs["new"]
        print(f"✅ KNOWN_IMG_DIR: {KNOWN_IMG_DIR}")
        print(f"✅ NEW_IMG_DIR:   {NEW_IMG_DIR}")

# Locate the previous run's zip (re-uploaded as a dataset) to salvage checkpoints
_prev_run_matches = list(Path("/kaggle/input").rglob("kan_qwen_shared28_best.pt"))
PREV_RUN_BASE = _prev_run_matches[0].parent if _prev_run_matches else None
print(f"{'✅' if PREV_RUN_BASE else '⚠️ '} Previous run (checkpoint salvage): {PREV_RUN_BASE}")

OUT = Path("/kaggle/working/part4_final")
for sub in ["", "part4_predictions", "confusion_matrices", "tables", "figures", "checkpoints_by_seed"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

if not Path("/kaggle/working/fast-kan").exists():
    os.system("git clone https://github.com/ZiyaoLi/fast-kan.git /kaggle/working/fast-kan")
sys.path.append("/kaggle/working/fast-kan")
from fastkan import FastKAN

with open(KAN_QWEN / "augmented_dataset_final.json") as f: qwen_data = json.load(f)
with open(KAN_LLAMA / "raw_dataset_FINAL.json") as f: llama_data = json.load(f)
qwen_text = np.load(KAN_QWEN / "aug_text_embeddings.npy")
qc = np.load(KAN_QWEN / "clip_features.npz", allow_pickle=True)
qwen_clip = {int(k): v for k, v in qc[qc.files[0]].item().items()}
qwen_clip_vecs = np.stack([qwen_clip[int(r["img_id"])] for r in qwen_data])
X_qwen_896_orig = np.concatenate([qwen_text, qwen_clip_vecs], axis=1).astype(np.float32)
llama_text = np.load(KAN_LLAMA / "text_embeddings.npy")
lc = np.load(KAN_LLAMA / "clip_features.npz", allow_pickle=True)
llama_clip = {int(k): v for k, v in lc[lc.files[0]].item().items()}
llama_clip_vecs = np.stack([llama_clip[int(r["img_id"])] for r in llama_data])
X_llama_896 = np.concatenate([llama_text, llama_clip_vecs], axis=1).astype(np.float32)
print(f"Qwen (pre-augmentation): {X_qwen_896_orig.shape}  Llama (pre-augmentation): {X_llama_896.shape}")


# --------------------------------------------------------------------------
# Item 1 — Shared-28 taxonomy
# --------------------------------------------------------------------------
qwen_classes_all = sorted(set(r["tool"] for r in qwen_data))
llama_classes_all = sorted(set(r["tool"] for r in llama_data))
shared_29_raw = sorted(set(qwen_classes_all) & set(llama_classes_all))
assert len(shared_29_raw) == 29
shared_28 = sorted([c for c in shared_29_raw if c != "matting"])
shared28_set = set(shared_28)
shared_28_mapping = {"class_to_local_id": {name: i for i, name in enumerate(shared_28)},
                      "local_id_to_class": {i: name for i, name in enumerate(shared_28)}}
local_id_to_class = shared_28_mapping["local_id_to_class"]
with open(OUT / "shared_28_class_mapping.json", "w") as f: json.dump(shared_28_mapping, f, indent=2)
with open(OUT / "shared28_taxonomy.txt", "w") as f: f.write("\n".join(f"{i}: {n}" for i, n in enumerate(shared_28)))
llama_matting_count = sum(1 for r in llama_data if r["tool"] == "matting")
print(f"✅ Shared-28 taxonomy: {len(shared_28)} classes")


# --------------------------------------------------------------------------
# Item 2 — human_shared28.csv
# --------------------------------------------------------------------------
def filter_acceptable_labels_to_shared28(label_str, s28):
    if pd.isna(label_str) or str(label_str).strip() == "": return ""
    return ", ".join([l.strip() for l in str(label_str).split(",") if l.strip() in s28])

human_shared28 = human_df[human_df["tool_name"].isin(shared28_set)].copy()
human_shared28["acceptable_tool_labels"] = human_shared28["acceptable_tool_labels"].apply(
    lambda x: filter_acceptable_labels_to_shared28(x, shared28_set))
human_shared28.to_csv(OUT / "human_shared28.csv", index=False)
has_multi = human_shared28["acceptable_tool_labels"].astype(str).str.strip() != ""
strict_true = human_shared28["strict_eval"] == True
human_strict_mask = strict_true & ~has_multi
human_ambig_mask = strict_true & has_multi
print(f"✅ human_shared28: {len(human_shared28)} rows")


# --------------------------------------------------------------------------
# Item 2.5 — Qwen corrective augmentation (deterministic, reproduces prior run)
# --------------------------------------------------------------------------
def normalize_q(q): return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x: self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry: self.parent[rx] = ry

def build_groups(records):
    n = len(records); uf = UnionFind(n)
    norm_to_idx, imgtool_to_idx = {}, {}
    for i, r in enumerate(records):
        norm = normalize_q(r["query"])
        if norm in norm_to_idx: uf.union(i, norm_to_idx[norm])
        else: norm_to_idx[norm] = i
    for i, r in enumerate(records):
        key = (r["img_id"], r["tool"])
        if key in imgtool_to_idx: uf.union(i, imgtool_to_idx[key])
        else: imgtool_to_idx[key] = i
    groups = {}
    for i in range(n): groups.setdefault(uf.find(i), []).append(i)
    return list(groups.values())

qwen_28_counts_raw = pd.Series([r["tool"] for r in qwen_data if r["tool"] in shared28_set]).value_counts()
starved_in_shared28 = qwen_28_counts_raw[qwen_28_counts_raw < 100].sort_values()
STARVED_FIX_CLASSES = list(starved_in_shared28.index)

from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

if STARVED_FIX_CLASSES:
    TARGET_PER_CLASS = 45
    PERSON_PROXY_TOOLS = {"person_detection", "action_recognition", "age_gender_estimation"}
    person_img_ids = sorted(set(r["img_id"] for r in qwen_data if r["tool"] in PERSON_PROXY_TOOLS))
    BASE_TEMPLATES = {
        "pose_estimation": ["Estimate the body pose of the person in this {noun}.",
            "What position is this person's body in, based on the {noun}?",
            "Map the skeletal keypoints of the person shown in the {noun}.",
            "Describe the pose this person is striking in the {noun}.",
            "Identify the joint locations and posture of the person visible in the {noun}."],
        "face_detection": ["Detect all faces in this {noun}.", "Locate every face visible in this {noun}.",
            "Find and mark the faces present in the {noun}.", "How many faces can you identify in this {noun}?",
            "Draw bounding boxes around each face in the {noun}."],
        "facial_expression_recognition": ["What facial expression is this person showing in the {noun}?",
            "Identify the emotion on this person's face in the {noun}.",
            "Is the person smiling, frowning, or neutral in this {noun}?",
            "Classify the facial expression visible in the {noun}.",
            "What mood does this person's face convey in the {noun}?"],
    }
    OPENERS = ["", "Please ", "Could you ", "I'd like you to ", "Go ahead and ", "Take a look and ",
               "Kindly ", "Now, ", "Next, ", "Also, ", "For this one, ", "Here, "]
    NOUNS = ["image", "photo", "picture", "photograph", "snapshot", "frame", "shot", "still",
             "scene", "visual", "capture", "graphic"]

    def make_unique_query(base_template, img_rank, class_rank):
        opener = OPENERS[img_rank % len(OPENERS)]
        noun = NOUNS[(img_rank + class_rank) % len(NOUNS)]
        text = base_template.format(noun=noun)
        return (opener + text[0].lower() + text[1:]) if opener else text

    rng = np.random.RandomState(SEED)
    new_rows, new_texts = [], []
    for cls_rank, cls in enumerate(STARVED_FIX_CLASSES):
        if cls not in BASE_TEMPLATES:
            continue
        pool = [i for i in person_img_ids if i in qwen_clip]
        rng.shuffle(pool)
        n_templates = len(BASE_TEMPLATES[cls])
        chosen_imgs = pool[:-(-TARGET_PER_CLASS // n_templates)]
        added, seen = 0, set()
        for img_rank, img_id in enumerate(chosen_imgs):
            for tmpl in BASE_TEMPLATES[cls]:
                if added >= TARGET_PER_CLASS: break
                q = make_unique_query(tmpl, img_rank, cls_rank)
                seen.add(normalize_q(q))
                new_rows.append({"img_id": img_id, "tool": cls, "query": q, "objects": None,
                                  "source": "manual_corrective_augmentation"})
                new_texts.append(q); added += 1
            if added >= TARGET_PER_CLASS: break
        print(f"  {cls}: +{added} rows across {len(chosen_imgs)} images")

    new_text_embs = minilm.encode(new_texts, batch_size=64, convert_to_numpy=True,
                                   normalize_embeddings=False).astype(np.float32)
    new_clip_vecs = np.stack([qwen_clip[r["img_id"]] for r in new_rows]).astype(np.float32)
    X_new = np.concatenate([new_text_embs, new_clip_vecs], axis=1).astype(np.float32)
    qwen_data = qwen_data + new_rows
    X_qwen_896_orig = np.concatenate([X_qwen_896_orig, X_new], axis=0)
    print(f"✅ qwen_data patched: {len(qwen_data)} rows")


# --------------------------------------------------------------------------
# Item 2.5b — Llama corrective augmentation
# --------------------------------------------------------------------------
llama_28_counts_raw = pd.Series([r["tool"] for r in llama_data if r["tool"] in shared28_set]).value_counts()
llama_starved = llama_28_counts_raw[llama_28_counts_raw < 5].sort_values()  # 3d_reconstruction had n=2
LLAMA_STARVED_FIX_CLASSES = list(llama_starved.index)

if LLAMA_STARVED_FIX_CLASSES:
    TARGET_PER_CLASS_LLAMA = 45
    llama_candidate_imgs = sorted(set(r["img_id"] for r in llama_data if r["img_id"] in llama_clip))
    BASE_TEMPLATES_LLAMA = {
        "3d_reconstruction": ["Reconstruct the 3D structure of the scene in this {noun}.",
            "Estimate the 3D geometry of the objects shown in this {noun}.",
            "Build a 3D model from this {noun}.", "What is the spatial 3D layout of this {noun}?",
            "Generate a 3D representation based on this {noun}."],
    }
    rng2 = np.random.RandomState(SEED)
    new_rows_l, new_texts_l = [], []
    for cls_rank, cls in enumerate(LLAMA_STARVED_FIX_CLASSES):
        if cls not in BASE_TEMPLATES_LLAMA:
            continue
        existing = set(r["img_id"] for r in llama_data if r["tool"] == cls)
        pool = [i for i in llama_candidate_imgs if i not in existing]
        rng2.shuffle(pool)
        n_templates = len(BASE_TEMPLATES_LLAMA[cls])
        chosen_imgs = pool[:-(-TARGET_PER_CLASS_LLAMA // n_templates)]
        added = 0
        for img_rank, img_id in enumerate(chosen_imgs):
            for tmpl in BASE_TEMPLATES_LLAMA[cls]:
                if added >= TARGET_PER_CLASS_LLAMA: break
                opener = OPENERS[img_rank % len(OPENERS)]
                noun = NOUNS[(img_rank + cls_rank) % len(NOUNS)]
                text = tmpl.format(noun=noun)
                q = (opener + text[0].lower() + text[1:]) if opener else text
                new_rows_l.append({"img_id": img_id, "tool": cls, "query": q, "objects": None,
                                    "source": "manual_corrective_augmentation"})
                new_texts_l.append(q); added += 1
            if added >= TARGET_PER_CLASS_LLAMA: break
        print(f"  {cls}: +{added} rows across {len(chosen_imgs)} images")

    new_text_embs_l = minilm.encode(new_texts_l, batch_size=64, convert_to_numpy=True,
                                     normalize_embeddings=False).astype(np.float32)
    new_clip_vecs_l = np.stack([llama_clip[r["img_id"]] for r in new_rows_l]).astype(np.float32)
    X_new_l = np.concatenate([new_text_embs_l, new_clip_vecs_l], axis=1).astype(np.float32)
    llama_data = llama_data + new_rows_l
    X_llama_896 = np.concatenate([X_llama_896, X_new_l], axis=0)
    print(f"✅ llama_data patched: {len(llama_data)} rows")


# --------------------------------------------------------------------------
# Items 3-4 — Leakage-safe splits + coverage check
# --------------------------------------------------------------------------
def group_split(groups, y_local, seed=SEED, train_frac=0.80, val_frac=0.10):
    rng = np.random.RandomState(seed)
    group_order = sorted(range(len(groups)), key=lambda gi: len(groups[gi]))
    used = set(); val_idx, test_idx = [], []
    all_classes = set(y_local.tolist())
    def smallest_avail(cls, excl):
        c = [(gi, len(groups[gi])) for gi in group_order
             if not any(i in excl for i in groups[gi]) and any(y_local[i] == cls for i in groups[gi])]
        return min(c, key=lambda x: x[1])[0] if c else None
    for cls in all_classes:
        if any(y_local[i] == cls for i in val_idx): continue
        gi = smallest_avail(cls, used)
        if gi is not None: val_idx.extend(groups[gi]); used.update(groups[gi])
    for cls in all_classes:
        if any(y_local[i] == cls for i in test_idx): continue
        gi = smallest_avail(cls, used)
        if gi is not None: test_idx.extend(groups[gi]); used.update(groups[gi])
    remaining = [gi for gi in group_order if not any(i in used for i in groups[gi])]
    rng.shuffle(remaining)
    n_tot = len(y_local); n_tr_t = int(train_frac * n_tot); n_va_t = int(val_frac * n_tot)
    train_idx = []
    for gi in remaining:
        idxs = groups[gi]
        if len(train_idx) < n_tr_t: train_idx.extend(idxs)
        elif len(val_idx) < n_va_t: val_idx.extend(idxs)
        else: test_idx.extend(idxs)
    return train_idx, val_idx, test_idx

def verify_leakage(idx_tr, idx_va, idx_te, records, name):
    def norms(idxs): return set(normalize_q(records[i]["query"]) for i in idxs)
    def imgtool(idxs): return set((records[i]["img_id"], records[i]["tool"]) for i in idxs)
    ntr, nva, nte = norms(idx_tr), norms(idx_va), norms(idx_te)
    itr, iva, ite = imgtool(idx_tr), imgtool(idx_va), imgtool(idx_te)
    r = {"norm_tr_va": len(ntr & nva), "norm_tr_te": len(ntr & nte), "norm_va_te": len(nva & nte),
         "imgtool_tr_va": len(itr & iva), "imgtool_tr_te": len(itr & ite), "imgtool_va_te": len(iva & ite)}
    print(f"=== {name} leakage ===")
    for k, v in r.items(): print(f"  {'✅' if v == 0 else '❌'} {k}: {v}")
    return r, all(v == 0 for v in r.values())

qwen_28_idx = [i for i, r in enumerate(qwen_data) if r["tool"] in shared28_set]
llama_28_idx = [i for i, r in enumerate(llama_data) if r["tool"] in shared28_set]
qwen_28_records = [qwen_data[i] for i in qwen_28_idx]
llama_28_records = [llama_data[i] for i in llama_28_idx]
X_qwen_28 = X_qwen_896_orig[qwen_28_idx]
X_llama_28 = X_llama_896[llama_28_idx]
y_qwen_28 = np.array([shared_28_mapping["class_to_local_id"][r["tool"]] for r in qwen_28_records], dtype=np.int64)
y_llama_28 = np.array([shared_28_mapping["class_to_local_id"][r["tool"]] for r in llama_28_records], dtype=np.int64)

qwen_groups_28 = build_groups(qwen_28_records)
llama_groups_28 = build_groups(llama_28_records)
idx_tr_Q28, idx_va_Q28, idx_te_Q28 = group_split(qwen_groups_28, y_qwen_28)
idx_tr_L28, idx_va_L28, idx_te_L28 = group_split(llama_groups_28, y_llama_28)

q_leak, q_zero = verify_leakage(idx_tr_Q28, idx_va_Q28, idx_te_Q28, qwen_28_records, "Qwen-28")
l_leak, l_zero = verify_leakage(idx_tr_L28, idx_va_L28, idx_te_L28, llama_28_records, "Llama-28")

all28 = set(range(28))
train_classes_Q, val_classes_Q, test_classes_Q = (set(y_qwen_28[i] for i in idx) for idx in (idx_tr_Q28, idx_va_Q28, idx_te_Q28))
train_classes_L, val_classes_L, test_classes_L = (set(y_llama_28[i] for i in idx) for idx in (idx_tr_L28, idx_va_L28, idx_te_L28))
assert q_zero and l_zero
assert train_classes_Q == all28 and val_classes_Q == all28 and test_classes_Q == all28
assert train_classes_L == all28 and val_classes_L == all28 and test_classes_L == all28
print("✅ Splits rebuilt: leakage-free, 28/28 coverage everywhere (deterministic reproduction)")

# NEW: cross-teacher mixed-training leak check + fix
def id_tuple(r): return (r["img_id"], r["tool"], normalize_q(r["query"]))
qwen_test_ids = set(id_tuple(r) for r in [qwen_28_records[i] for i in idx_te_Q28])
llama_test_ids = set(id_tuple(r) for r in [llama_28_records[i] for i in idx_te_L28])
idx_tr_Q28_clean = [i for i in idx_tr_Q28 if id_tuple(qwen_28_records[i]) not in llama_test_ids]
idx_tr_L28_clean = [i for i in idx_tr_L28 if id_tuple(llama_28_records[i]) not in qwen_test_ids]
n_removed_q = len(idx_tr_Q28) - len(idx_tr_Q28_clean)
n_removed_l = len(idx_tr_L28) - len(idx_tr_L28_clean)
print(f"Cross-teacher mixed-training leak fix: removed {n_removed_q} Qwen-train + "
      f"{n_removed_l} Llama-train rows that duplicated the OTHER teacher's test set")


# --------------------------------------------------------------------------
# Item 5 — Feature verification + human features
# --------------------------------------------------------------------------
import open_clip
from PIL import Image

def find_coco_image(img_id, image_dirs):
    for d in image_dirs:
        p = Path(d) / f"{img_id}.jpg"
        if p.exists(): return p
    return None

image_dirs = [KNOWN_IMG_DIR, NEW_IMG_DIR]
caches = {**qwen_clip, **llama_clip}
candidates = [i for i in caches.keys() if find_coco_image(i, image_dirs) is not None][:15]
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai", force_quick_gelu=True)
clip_model = clip_model.to(DEVICE).eval()
diffs = []
for img_id in candidates:
    image = Image.open(find_coco_image(img_id, image_dirs)).convert("RGB")
    pixel_values = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        fresh = clip_model.encode_image(pixel_values)
        fresh = (fresh / fresh.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
    diffs.append(float(np.abs(fresh - caches[img_id]).max()))
provenance_result = {"images_tested": len(candidates), "maximum_absolute_difference": max(diffs) if diffs else None,
                      "PROVENANCE_VERIFIED": bool(diffs) and max(diffs) < 1e-4}
assert provenance_result["PROVENANCE_VERIFIED"]
print(f"✅ Provenance verified: max_diff={provenance_result['maximum_absolute_difference']:.2e}")

human_text_28 = minilm.encode(human_shared28["query"].tolist(), batch_size=256,
                               convert_to_numpy=True, normalize_embeddings=False).astype(np.float32)
human_clip_vecs = []
for iid in human_shared28["image_id"]:
    p = find_coco_image(iid, image_dirs)
    image = Image.open(p).convert("RGB")
    pixel_values = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        f = clip_model.encode_image(pixel_values)
        f = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
    human_clip_vecs.append(f)
human_clip_vecs = np.stack(human_clip_vecs).astype(np.float32)
X_human_28 = np.concatenate([human_text_28, human_clip_vecs], axis=1).astype(np.float32)
y_human_28_canonical = human_shared28["tool_name"].map(shared_28_mapping["class_to_local_id"]).values.astype(np.int64)
print(f"✅ Human features: {X_human_28.shape}")


# --------------------------------------------------------------------------
# Cell 1 — Architecture
# --------------------------------------------------------------------------
class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=28, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim); self.norm = nn.LayerNorm(proj_dim)
        self.kan = FastKAN(layers_hidden=[proj_dim] + hidden_dims + [num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=[64, 32], num_tools=28, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class ToolRoutingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def evaluate_full_probs(model, loader):
    model.eval(); preds, labels, all_probs = [], [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            logits = model(X_b); probs = torch.softmax(logits, dim=-1)
            preds.extend(logits.argmax(-1).cpu().numpy()); labels.extend(y_b.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(preds), np.array(labels), np.array(all_probs)

def train_model(model, tr, va, name, epochs=60, lr=1e-3, wd=1e-4, is_kan=False, seed=42, save_dir=None):
    torch.manual_seed(seed)
    if is_kan: lr, epochs = 5e-4, 80
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(tr), epochs=epochs, pct_start=0.1)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, path = 0.0, (save_dir or OUT) / f"{name}_best.pt"
    for ep in range(1, epochs + 1):
        model.train()
        for X_b, y_b in tr:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        model.eval(); c, t = 0, 0
        with torch.no_grad():
            for X_b, y_b in va:
                X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
                c += (model(X_b).argmax(-1) == y_b).sum().item(); t += len(y_b)
        acc = c / t
        if acc > best: best = acc; torch.save(model.state_dict(), path)
        if ep % 20 == 0 or ep == 1: print(f"  [{name}] ep{ep}/{epochs} val={acc:.4f}")
    model.load_state_dict(torch.load(path))
    return best

TRAINING_CONFIG = dict(optimizer="AdamW", lr_mlp=1e-3, lr_kan=5e-4, weight_decay=1e-4, batch_size=256,
                        epochs_mlp=60, epochs_kan=80, scheduler="OneCycleLR", grad_clip=1.0,
                        checkpoint_criterion="best_val_accuracy")
SEEDS = [42, 123, 2026]
CKPT_DIR = OUT / "checkpoints_by_seed"

def make_loader(X, y, batch_size=256, shuffle=True):
    return DataLoader(ToolRoutingDataset(X, y), batch_size=batch_size, shuffle=shuffle)


# --------------------------------------------------------------------------
# Item 6 — Regimes (Mixed now uses the leak-free training pool)
# --------------------------------------------------------------------------
X_tr_Q, y_tr_Q = X_qwen_28[idx_tr_Q28], y_qwen_28[idx_tr_Q28]
X_va_Q, y_va_Q = X_qwen_28[idx_va_Q28], y_qwen_28[idx_va_Q28]
X_te_Q, y_te_Q = X_qwen_28[idx_te_Q28], y_qwen_28[idx_te_Q28]
X_tr_L, y_tr_L = X_llama_28[idx_tr_L28], y_llama_28[idx_tr_L28]
X_va_L, y_va_L = X_llama_28[idx_va_L28], y_llama_28[idx_va_L28]
X_te_L, y_te_L = X_llama_28[idx_te_L28], y_llama_28[idx_te_L28]

X_tr_Mix = np.concatenate([X_qwen_28[idx_tr_Q28_clean], X_llama_28[idx_tr_L28_clean]], axis=0)
y_tr_Mix = np.concatenate([y_qwen_28[idx_tr_Q28_clean], y_llama_28[idx_tr_L28_clean]], axis=0)
X_va_Mix = np.concatenate([X_va_Q, X_va_L], axis=0)
y_va_Mix = np.concatenate([y_va_Q, y_va_L], axis=0)
print(f"Mixed train (leak-free): {len(y_tr_Mix)} rows (was {len(y_tr_Q)+len(y_tr_L)} before the fix)")

eval_loaders = {"qwen_test": make_loader(X_te_Q, y_te_Q, shuffle=False),
                "llama_test": make_loader(X_te_L, y_te_L, shuffle=False),
                "human": make_loader(X_human_28, y_human_28_canonical, shuffle=False)}


# --------------------------------------------------------------------------
# Items 7-8 — Load 12 valid checkpoints, retrain only the 6 mixed models
# --------------------------------------------------------------------------
all_models, train_log_rows = {}, []

for regime, X_tr, y_tr in [("qwen", X_tr_Q, y_tr_Q), ("llama", X_tr_L, y_tr_L)]:
    for arch in ["kan", "mlp"]:
        for seed in SEEDS:
            name = f"{arch}_{regime}_shared28_seed{seed}"
            ckpt_src = None
            if PREV_RUN_BASE:
                candidate = PREV_RUN_BASE / "checkpoints_by_seed" / f"{name}_best.pt"
                if candidate.exists(): ckpt_src = candidate
                elif seed == 42:
                    canon = PREV_RUN_BASE / f"{arch}_{regime}_shared28_best.pt"
                    if canon.exists(): ckpt_src = canon
            if ckpt_src:
                dst = CKPT_DIR / f"{name}_best.pt"
                shutil.copy(ckpt_src, dst)
                all_models[(regime, arch, seed)] = dst
                print(f"  ♻️  {name}: reused from previous run")
            else:
                print(f"  ⚠️  {name}: not found in previous run, training fresh")
                tr_loader = make_loader(X_tr, y_tr)
                va_loader = make_loader(X_va_Q if regime == "qwen" else X_va_L,
                                         y_va_Q if regime == "qwen" else y_va_L, shuffle=False)
                model = (KANRoute if arch == "kan" else MLPBaseline)(input_dim=896, num_tools=28).to(DEVICE)
                best_val = train_model(model, tr_loader, va_loader, name, is_kan=(arch == "kan"),
                                        seed=seed, save_dir=CKPT_DIR)
                all_models[(regime, arch, seed)] = CKPT_DIR / f"{name}_best.pt"
                train_log_rows.append(dict(regime=regime, arch=arch, seed=seed, best_val_acc=best_val))

mixed_tr_loader = make_loader(X_tr_Mix, y_tr_Mix)
mixed_va_loader = make_loader(X_va_Mix, y_va_Mix, shuffle=False)
for arch in ["kan", "mlp"]:
    for seed in SEEDS:
        name = f"{arch}_mixed_shared28_seed{seed}"
        model = (KANRoute if arch == "kan" else MLPBaseline)(input_dim=896, num_tools=28).to(DEVICE)
        best_val = train_model(model, mixed_tr_loader, mixed_va_loader, name,
                                is_kan=(arch == "kan"), seed=seed, save_dir=CKPT_DIR)
        all_models[("mixed", arch, seed)] = CKPT_DIR / f"{name}_best.pt"
        n_params = model.count_parameters()
        train_log_rows.append(dict(regime="mixed", arch=arch, seed=seed, best_val_acc=best_val, n_params=n_params))
        print(f"  ✅ {name}: best_val_acc={best_val:.4f}")

train_log = pd.DataFrame(train_log_rows)
train_log.to_csv(OUT / "part4_training_log.csv", index=False)

CANON_NAMES = {("qwen", "kan"): "kan_qwen_shared28_best.pt", ("qwen", "mlp"): "mlp_qwen_shared28_best.pt",
               ("llama", "kan"): "kan_llama_shared28_best.pt", ("llama", "mlp"): "mlp_llama_shared28_best.pt",
               ("mixed", "kan"): "kan_mixed_shared28_best.pt", ("mixed", "mlp"): "mlp_mixed_shared28_best.pt"}
for (regime, arch), fname in CANON_NAMES.items():
    shutil.copy(all_models[(regime, arch, 42)], OUT / fname)
print("✅ Canonical checkpoints ready (12 reused + 6 freshly retrained)")

with open(OUT / "mixed_training_leak_fix_provenance.md", "w") as f:
    f.write("# Mixed-Training Cross-Teacher Leak Fix\n\n"
            "A 15th sanity check (not in the original prompt's Item 20 list, added during review) "
            "found that leakage-safe grouping was computed independently per teacher, so it never "
            "checked whether a Qwen test row and a Llama train row (or vice versa) could be "
            "near-identical (same image, same tool, same/similar phrasing) due to cross-teacher "
            f"overlap in the underlying COCO image pool.\n\nFound: {n_removed_q} Qwen-train row(s) "
            f"duplicating a Llama-test row, {n_removed_l} Llama-train row(s) duplicating a "
            f"Qwen-test row (out of ~38,000 combined training rows — negligible in scale, but "
            f"fixed rather than left in place). These rows were excluded from the Mixed regime's "
            f"training pool only; Qwen-only and Llama-only training/checkpoints/results are "
            f"unaffected and were reused unchanged from the prior run.\n")
print("✅ mixed_training_leak_fix_provenance.md written")

In [ ]:
##############################################################################
# RESUME SCRIPT PART 2 — run immediately after Part 1 completes.
##############################################################################

# --------------------------------------------------------------------------
# Item 9 (enriched from the start) — full evaluation matrix
# --------------------------------------------------------------------------
def top_k_acc(probs, labels, k):
    topk = np.argsort(-probs, axis=1)[:, :k]
    return np.mean([labels[i] in topk[i] for i in range(len(labels))])

def compute_metrics(preds, labels, probs, num_classes=28):
    return dict(top1=np.mean(preds == labels), top3=top_k_acc(probs, labels, 3),
                macro_f1=f1_score(labels, preds, average="macro", labels=list(range(num_classes)), zero_division=0),
                weighted_f1=f1_score(labels, preds, average="weighted", labels=list(range(num_classes)), zero_division=0))

def ambiguity_aware_score(pred_top1, pred_top3_names, acceptable_labels_str):
    pred_name = local_id_to_class[pred_top1]
    acceptable = set(x.strip() for x in str(acceptable_labels_str).split(",") if x.strip())
    return (pred_name in acceptable), bool(set(pred_top3_names) & acceptable)

PRED_DIR = OUT / "part4_predictions"
results_rows, per_class_metrics_rows = [], []

for (regime, arch, seed), ckpt_path in all_models.items():
    model = (KANRoute if arch == "kan" else MLPBaseline)(input_dim=896, num_tools=28).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path)); model.eval()

    for eval_name, loader in eval_loaders.items():
        preds, labels, probs = evaluate_full_probs(model, loader)
        top3_preds = np.argsort(-probs, axis=1)[:, :3]
        probs_str = [";".join(f"{p:.6f}" for p in row) for row in probs]

        if eval_name == "qwen_test":
            src_records, record_idx = [qwen_28_records[i] for i in idx_te_Q28], list(idx_te_Q28)
        elif eval_name == "llama_test":
            src_records, record_idx = [llama_28_records[i] for i in idx_te_L28], list(idx_te_L28)
        else:
            src_records, record_idx = human_shared28.to_dict("records"), human_shared28["query_id"].tolist()

        pred_df = pd.DataFrame({
            "sample_id": record_idx,
            "img_id": [r.get("img_id", r.get("image_id")) for r in src_records],
            "query": [r.get("query") for r in src_records],
            "ground_truth_tool": [r.get("tool", r.get("tool_name")) for r in src_records],
            "ground_truth": labels, "pred_top1": preds,
            "pred_top1_tool": [local_id_to_class[p] for p in preds],
            "pred_top3": [list(map(int, t)) for t in top3_preds], "probs": probs_str,
            "correct": (preds == labels), "training_source": regime, "evaluation_source": eval_name,
            "architecture": arch, "seed": seed,
        })
        pred_df.to_csv(PRED_DIR / f"{arch}_{regime}_seed{seed}_on_{eval_name}.csv", index=False)

        p, r_, f, s = precision_recall_fscore_support(labels, preds, labels=list(range(28)), zero_division=0)
        for cls_id in range(28):
            per_class_metrics_rows.append(dict(training_source=regime, architecture=arch, seed=seed,
                evaluation_source=eval_name, class_id=cls_id, class_name=local_id_to_class[cls_id],
                precision=p[cls_id], recall=r_[cls_id], f1=f[cls_id], support=int(s[cls_id])))

        if eval_name != "human":
            m = compute_metrics(preds, labels, probs, 28)
            results_rows.append(dict(training_source=regime, architecture=arch, seed=seed, evaluation_source=eval_name, **m))
        else:
            m_canon = compute_metrics(preds, labels, probs, 28)
            sm = human_strict_mask.values
            m_strict = compute_metrics(preds[sm], labels[sm], probs[sm], 28) if sm.sum() else dict(top1=np.nan, top3=np.nan, macro_f1=np.nan, weighted_f1=np.nan)
            am = human_ambig_mask.values
            if am.sum():
                acc_labels = human_shared28.loc[am, "acceptable_tool_labels"].values
                t1h, t3h = [], []
                for i, row_i in enumerate(np.where(am)[0]):
                    a, b = ambiguity_aware_score(preds[row_i], [local_id_to_class[c] for c in top3_preds[row_i]], acc_labels[i])
                    t1h.append(a); t3h.append(b)
                ambig_top1, ambig_top3 = float(np.mean(t1h)), float(np.mean(t3h))
            else:
                ambig_top1 = ambig_top3 = np.nan
            results_rows.append(dict(training_source=regime, architecture=arch, seed=seed, evaluation_source="human",
                top1=m_canon["top1"], top3=m_canon["top3"], macro_f1=m_canon["macro_f1"], weighted_f1=m_canon["weighted_f1"],
                strict_top1=m_strict["top1"], strict_top3=m_strict["top3"], strict_macro_f1=m_strict["macro_f1"],
                strict_weighted_f1=m_strict["weighted_f1"], ambiguity_aware_top1=ambig_top1, ambiguity_aware_top3=ambig_top3))

part4_all_results = pd.DataFrame(results_rows)
part4_all_results.to_csv(OUT / "part4_all_results.csv", index=False)
part4_per_class_metrics = pd.DataFrame(per_class_metrics_rows)
part4_per_class_metrics.to_csv(OUT / "part4_per_class_metrics.csv", index=False)
print(f"✅ part4_all_results.csv: {part4_all_results.shape}, part4_per_class_metrics.csv: {part4_per_class_metrics.shape}")
print(part4_all_results.groupby(["training_source", "architecture", "evaluation_source"])[["top1", "macro_f1"]].mean())


# --------------------------------------------------------------------------
# Save the 6 required split CSVs
# --------------------------------------------------------------------------
def records_to_df(records, idxs): return pd.DataFrame([records[i] for i in idxs])
pd.DataFrame([qwen_28_records[i] for i in idx_tr_Q28]).to_csv(OUT / "qwen_shared28_train.csv", index=False)
pd.DataFrame([qwen_28_records[i] for i in idx_va_Q28]).to_csv(OUT / "qwen_shared28_val.csv", index=False)
pd.DataFrame([qwen_28_records[i] for i in idx_te_Q28]).to_csv(OUT / "qwen_shared28_test.csv", index=False)
pd.DataFrame([llama_28_records[i] for i in idx_tr_L28]).to_csv(OUT / "llama_shared28_train.csv", index=False)
pd.DataFrame([llama_28_records[i] for i in idx_va_L28]).to_csv(OUT / "llama_shared28_val.csv", index=False)
pd.DataFrame([llama_28_records[i] for i in idx_te_L28]).to_csv(OUT / "llama_shared28_test.csv", index=False)
print("✅ 6 split CSVs saved")


# --------------------------------------------------------------------------
# Items 11-14 — distribution shift, recovery, human OOD, per-class transfer
# --------------------------------------------------------------------------
agg = part4_all_results[part4_all_results["evaluation_source"].isin(["qwen_test", "llama_test"])] \
    .groupby(["training_source", "architecture", "evaluation_source"])[["top1", "macro_f1"]].mean().reset_index()

drop_rows = []
for arch in ["kan", "mlp"]:
    for train_src, in_ev, cr_ev in [("qwen", "qwen_test", "llama_test"), ("llama", "llama_test", "qwen_test")]:
        i_ = agg[(agg.training_source == train_src) & (agg.architecture == arch) & (agg.evaluation_source == in_ev)]
        c_ = agg[(agg.training_source == train_src) & (agg.architecture == arch) & (agg.evaluation_source == cr_ev)]
        if len(i_) and len(c_):
            drop_rows.append(dict(model=f"{arch}_{train_src}", architecture=arch, training_source=train_src,
                in_domain_top1=i_.top1.values[0], cross_teacher_top1=c_.top1.values[0],
                top1_drop=i_.top1.values[0]-c_.top1.values[0], in_domain_macro_f1=i_.macro_f1.values[0],
                cross_teacher_macro_f1=c_.macro_f1.values[0], macro_f1_drop=i_.macro_f1.values[0]-c_.macro_f1.values[0]))
distribution_shift_df = pd.DataFrame(drop_rows)
distribution_shift_df.to_csv(OUT / "distribution_shift.csv", index=False)

human_agg = part4_all_results[part4_all_results.evaluation_source == "human"].groupby(["training_source", "architecture"])[
    ["top1", "strict_top1", "strict_top3", "macro_f1", "ambiguity_aware_top1", "ambiguity_aware_top3"]].mean().reset_index()
human_agg.to_csv(OUT / "human_ood_summary.csv", index=False)

def per_class_f1_from_preds(arch, ts, es, seed=42):
    df = pd.read_csv(PRED_DIR / f"{arch}_{ts}_seed{seed}_on_{es}.csv")
    _, _, f, _ = precision_recall_fscore_support(df["ground_truth"], df["pred_top1"], labels=list(range(28)), zero_division=0)
    return f

transfer_rows = []
for cls_id in range(28):
    row = {"class_id": cls_id, "class_name": local_id_to_class[cls_id]}
    row["qwen_to_qwen_f1"] = per_class_f1_from_preds("kan", "qwen", "qwen_test")[cls_id]
    row["qwen_to_llama_f1"] = per_class_f1_from_preds("kan", "qwen", "llama_test")[cls_id]
    row["llama_to_llama_f1"] = per_class_f1_from_preds("kan", "llama", "llama_test")[cls_id]
    row["llama_to_qwen_f1"] = per_class_f1_from_preds("kan", "llama", "qwen_test")[cls_id]
    row["mixed_to_qwen_f1"] = per_class_f1_from_preds("kan", "mixed", "qwen_test")[cls_id]
    row["mixed_to_llama_f1"] = per_class_f1_from_preds("kan", "mixed", "llama_test")[cls_id]
    transfer_rows.append(row)
shared28_per_class_transfer = pd.DataFrame(transfer_rows)
shared28_per_class_transfer.to_csv(OUT / "shared28_per_class_transfer.csv", index=False)
print("✅ Items 11-14 done")
print(distribution_shift_df)
print(human_agg)


# --------------------------------------------------------------------------
# Item 15 — Confusion matrices
# --------------------------------------------------------------------------
import matplotlib.pyplot as plt
CM_DIR = OUT / "confusion_matrices"

def save_confmat(arch, ts, es, seed=42):
    df = pd.read_csv(PRED_DIR / f"{arch}_{ts}_seed{seed}_on_{es}.csv")
    cm = confusion_matrix(df["ground_truth"], df["pred_top1"], labels=list(range(28)), normalize="true")
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{arch.upper()} {ts}→{es} (normalized)"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout()
    fig.savefig(CM_DIR / f"{arch}_{ts}_to_{es}_confusion.png", dpi=150); plt.close(fig)

for arch, ts, es in [("kan","qwen","llama_test"),("kan","llama","qwen_test"),("kan","mixed","qwen_test"),
                     ("kan","mixed","llama_test"),("mlp","qwen","llama_test"),("mlp","llama","qwen_test"),
                     ("kan","qwen","human"),("kan","llama","human"),("kan","mixed","human")]:
    save_confmat(arch, ts, es)
print("✅ Confusion matrices saved")


# --------------------------------------------------------------------------
# Item 16 — Bootstrap CI
# --------------------------------------------------------------------------
def bootstrap_ci(preds, labels, metric_fn, n_boot=1000, seed=0):
    rng = np.random.RandomState(seed); n = len(labels); stats = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n); stats.append(metric_fn(preds[idx], labels[idx]))
    stats = np.array(stats)
    return float(np.mean(stats)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))

def top1_fn(p, l): return np.mean(p == l)
def macrof1_fn(p, l): return f1_score(l, p, average="macro", labels=list(range(28)), zero_division=0)

ci_rows = []
for arch in ["kan", "mlp"]:
    for ts, es in [("qwen","llama_test"),("llama","qwen_test"),("mixed","qwen_test"),("mixed","llama_test"),
                   ("qwen","human"),("llama","human"),("mixed","human")]:
        df = pd.read_csv(PRED_DIR / f"{arch}_{ts}_seed42_on_{es}.csv")
        preds, labels = df["pred_top1"].values, df["ground_truth"].values
        t1m, t1l, t1h = bootstrap_ci(preds, labels, top1_fn)
        fm, fl, fh = bootstrap_ci(preds, labels, macrof1_fn)
        ci_rows.append(dict(architecture=arch, training_source=ts, evaluation_source=es, top1_estimate=t1m,
                             top1_ci_low=t1l, top1_ci_high=t1h, macro_f1_estimate=fm, macro_f1_ci_low=fl, macro_f1_ci_high=fh))
part4_bootstrap_ci = pd.DataFrame(ci_rows)
part4_bootstrap_ci.to_csv(OUT / "part4_bootstrap_ci.csv", index=False)
print("✅ Bootstrap CIs saved")


# --------------------------------------------------------------------------
# Item 18 — Tables A-D
# --------------------------------------------------------------------------
TABLES_DIR = OUT / "tables"
def df_to_latex(df, cap, lab, path): 
    with open(path, "w") as f: f.write(df.to_latex(index=False, float_format="%.4f", caption=cap, label=lab, escape=True))

tableA_rows = []
for (ts, arch), g in agg.groupby(["training_source", "architecture"]):
    q, l = g[g.evaluation_source=="qwen_test"], g[g.evaluation_source=="llama_test"]
    if len(q) and len(l):
        tableA_rows.append(dict(Training=ts, Model=arch.upper(), Qwen_Top1=q.top1.values[0], Qwen_MacroF1=q.macro_f1.values[0],
                                 Llama_Top1=l.top1.values[0], Llama_MacroF1=l.macro_f1.values[0]))
tableA = pd.DataFrame(tableA_rows)
df_to_latex(tableA, "Cross-Teacher Generalization (Shared-28)", "tab:cross_teacher", TABLES_DIR/"table_cross_teacher.tex")

tableB = human_agg.rename(columns={"training_source":"Training","architecture":"Model","top1":"Canonical_Top1",
    "strict_top1":"Strict_Top1","strict_top3":"Strict_Top3","macro_f1":"Macro_F1",
    "ambiguity_aware_top1":"Ambig_Top1","ambiguity_aware_top3":"Ambig_Top3"})
tableB["Model"] = tableB["Model"].str.upper()
df_to_latex(tableB, "Human OOD Generalization (Shared-28)", "tab:human_ood", TABLES_DIR/"table_human_ood.tex")

tableC = distribution_shift_df.rename(columns={"architecture":"Model","training_source":"Source",
    "in_domain_top1":"In_domain","cross_teacher_top1":"Cross_teacher","top1_drop":"Top1_Drop","macro_f1_drop":"F1_Drop"}
    )[["Model","Source","In_domain","Cross_teacher","Top1_Drop","F1_Drop"]]
tableC["Model"] = tableC["Model"].str.upper()
df_to_latex(tableC, "Distribution Shift", "tab:dist_shift", TABLES_DIR/"table_distribution_shift.tex")

recovery_rows = []
for arch in ["kan", "mlp"]:
    for target, opp_src in [("qwen_test","llama"),("llama_test","qwen")]:
        o = agg[(agg.training_source==opp_src)&(agg.architecture==arch)&(agg.evaluation_source==target)]
        m = agg[(agg.training_source=="mixed")&(agg.architecture==arch)&(agg.evaluation_source==target)]
        if len(o) and len(m):
            recovery_rows.append(dict(Architecture=arch.upper(), Target=target, Single_opposite_teacher=o.top1.values[0],
                Mixed=m.top1.values[0], Recovery_Top1=m.top1.values[0]-o.top1.values[0], Recovery_Macro_F1=m.macro_f1.values[0]-o.macro_f1.values[0]))
tableD = pd.DataFrame(recovery_rows)
df_to_latex(tableD, "Multi-Teacher Recovery", "tab:multi_teacher_recovery", TABLES_DIR/"table_multi_teacher_recovery.tex")
print("✅ Tables A-D saved"); print(tableA); print(tableD)


# --------------------------------------------------------------------------
# Item 19 — Figures (including corrected single-file cross_teacher_generalization.png)
# --------------------------------------------------------------------------
FIG_DIR = OUT / "figures"

def get_top1(arch, ts, ev):
    r = agg[(agg.training_source==ts)&(agg.architecture==arch)&(agg.evaluation_source==ev)]
    return r.top1.values[0] if len(r) else np.nan

fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(3); width = 0.35
for i, arch in enumerate(["kan","mlp"]):
    in_domain = np.nanmean([get_top1(arch,"qwen","qwen_test"), get_top1(arch,"llama","llama_test")])
    cross = np.nanmean([get_top1(arch,"qwen","llama_test"), get_top1(arch,"llama","qwen_test")])
    mixed = np.nanmean([get_top1(arch,"mixed","qwen_test"), get_top1(arch,"mixed","llama_test")])
    ax.bar(x + i*width, [in_domain, cross, mixed], width, label=arch.upper())
ax.set_xticks(x + width/2); ax.set_xticklabels(["In-domain","Cross-teacher","Mixed"])
ax.set_ylabel("Top-1 Accuracy"); ax.set_ylim(0,1.0)
ax.set_title("Cross-Teacher Generalization"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR/"cross_teacher_generalization.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(8,5))
xs = np.arange(len(tableD)); ax.bar(xs-0.175, tableD.Single_opposite_teacher, 0.35, label="Single opposite teacher")
ax.bar(xs+0.175, tableD.Mixed, 0.35, label="Mixed")
ax.set_xticks(xs); ax.set_xticklabels([f"{r.Architecture} {r.Target}" for r in tableD.itertuples()])
ax.set_ylabel("Top-1 Accuracy"); ax.set_ylim(0,1.0); ax.set_title("Multi-Teacher Recovery"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR/"multi_teacher_recovery.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(8,5))
xs = np.arange(2); width=0.25
for i, ts in enumerate(["qwen","llama","mixed"]):
    vals = [human_agg[(human_agg.architecture==a)&(human_agg.training_source==ts)].top1.values[0] for a in ["kan","mlp"]]
    ax.bar(xs + i*width, vals, width, label=f"{ts.capitalize()}-trained")
ax.set_xticks(xs + width); ax.set_xticklabels(["KAN","MLP"])
ax.set_ylabel("Top-1 Accuracy"); ax.set_ylim(0,1.0)
ax.set_title("Human-Query OOD Generalization"); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR/"human_ood_generalization.png", dpi=150); plt.close(fig)
print("✅ Figures saved")


# --------------------------------------------------------------------------
# Item 20 — Full 15-check sanity gate
# --------------------------------------------------------------------------
_dummy = torch.randn(2, 896).to(DEVICE)
_kan_check = KANRoute(input_dim=896, num_tools=28).to(DEVICE); _kan_check.load_state_dict(torch.load(all_models[("qwen","kan",42)])); _kan_check.eval()
_mlp_check = MLPBaseline(input_dim=896, num_tools=28).to(DEVICE); _mlp_check.load_state_dict(torch.load(all_models[("qwen","mlp",42)])); _mlp_check.eval()
with torch.no_grad():
    dim_kan_ok = _kan_check(_dummy).shape[-1] == 28
    dim_mlp_ok = _mlp_check(_dummy).shape[-1] == 28

mixed_train_ids_final = set(id_tuple(r) for r in [qwen_28_records[i] for i in idx_tr_Q28_clean]) | \
                         set(id_tuple(r) for r in [llama_28_records[i] for i in idx_tr_L28_clean])
_human_pred_check = pd.read_csv(PRED_DIR / "kan_qwen_seed42_on_human.csv")

recompute_mismatch = max(
    abs(float(np.mean(pd.read_csv(PRED_DIR/f"kan_{ts}_seed42_on_{es}.csv")["correct"])) -
        part4_all_results[(part4_all_results.training_source==ts)&(part4_all_results.architecture=="kan")&
                           (part4_all_results.seed==42)&(part4_all_results.evaluation_source==es)]["top1"].values[0])
    for ts, es in [("qwen","qwen_test"),("llama","llama_test")]
)

full_sanity = {
    "[1] taxonomy contains exactly 28 classes": len(shared_28) == 28,
    "[2] matting excluded": "matting" not in shared_28,
    "[3] Qwen mapping == Llama mapping": True,  # single shared dict by construction
    "[4] KAN output dimension = 28": dim_kan_ok,
    "[5] MLP output dimension = 28": dim_mlp_ok,
    "[6] Qwen train/test normalized-query overlap = 0": q_leak["norm_tr_te"] == 0,
    "[7] Llama train/test normalized-query overlap = 0": l_leak["norm_tr_te"] == 0,
    "[8] Qwen train/test image-tool overlap = 0": q_leak["imgtool_tr_te"] == 0,
    "[9] Llama train/test image-tool overlap = 0": l_leak["imgtool_tr_te"] == 0,
    "[10] no Qwen test sample used during mixed training": len(qwen_test_ids & mixed_train_ids_final) == 0,
    "[11] no Llama test sample used during mixed training": len(llama_test_ids & mixed_train_ids_final) == 0,
    "[12] prediction counts equal evaluation-row counts": all(
        len(pd.read_csv(PRED_DIR/f"kan_{ts}_seed42_on_{es}.csv")) == n
        for ts, es, n in [("qwen","qwen_test",len(y_te_Q)),("llama","llama_test",len(y_te_L)),("qwen","human",len(y_human_28_canonical))]),
    "[13] human predictions aligned by query_id": _human_pred_check["sample_id"].tolist() == human_shared28["query_id"].tolist(),
    "[14] all reported metrics recomputed from saved predictions": recompute_mismatch < 1e-9,
    "[15] no NaN/Inf metrics": not (part4_all_results.select_dtypes("number").isin([np.inf,-np.inf]).any().any()),
}
print("\n=== Full Item 20 sanity checklist (15/15) ===")
all_pass = True
for k, v in full_sanity.items():
    print(f"  {'✅' if v else '❌'} {k}"); all_pass = all_pass and bool(v)
assert all_pass, "STOP: a sanity check failed"
print("✅ All 15/15 passed")

with open(OUT / "shared28_split_leakage_audit.md", "w") as f:
    f.write("# Shared-28 Split Leakage + Full Sanity Audit\n\n## Qwen-28\n\n")
    for k, v in q_leak.items(): f.write(f"- {k}: {v}\n")
    f.write(f"\n## Llama-28\n\n")
    for k, v in l_leak.items(): f.write(f"- {k}: {v}\n")
    f.write(f"\n## Full 15-check sanity gate\n\n")
    for k, v in full_sanity.items(): f.write(f"- {'PASS' if v else 'FAIL'}: {k}\n")
    f.write(f"\n## Mixed-training cross-teacher leak fix\n\n"
            f"Removed {n_removed_q} Qwen-train + {n_removed_l} Llama-train rows duplicating "
            f"the other teacher's test set. See mixed_training_leak_fix_provenance.md.\n")
print("✅ shared28_split_leakage_audit.md written")


# --------------------------------------------------------------------------
# Items 23-24 — Reports
# --------------------------------------------------------------------------
report = [f"# PART4_RESULTS_REPORT.md\n",
    f"## 1. Shared-28 taxonomy\nShared-29 minus matting ({len(shared_29_raw)}→{len(shared_28)}).\n",
    f"## 2. Dataset sizes\nQwen: train={len(y_tr_Q)} val={len(y_va_Q)} test={len(y_te_Q)}\n"
    f"Llama: train={len(y_tr_L)} val={len(y_va_L)} test={len(y_te_L)}\nMixed (leak-free): train={len(y_tr_Mix)}\n",
    f"## 3. Leakage + sanity\nAll 15/15 checks passed, including the cross-teacher mixed-training check.\n",
    f"## 4. Training config\n{TRAINING_CONFIG}\nSeeds: {SEEDS}\n",
    f"## 5. In-domain/cross-teacher\n{agg.to_markdown(index=False)}\n",
    f"## 6. Distribution shift\n{distribution_shift_df.to_markdown(index=False)}\n",
    f"## 7. Multi-teacher recovery\n{tableD.to_markdown(index=False)}\n",
    f"## 8. Human OOD\n{human_agg.to_markdown(index=False)}\n",
    f"## 9. Mixed-training leak fix\nRemoved {n_removed_q}+{n_removed_l} cross-teacher-duplicate rows "
    f"from mixed training. Impact expected to be negligible given scale (~38k rows).\n",
    f"## 10. Limitations\n- Manual corrective augmentation for pose_estimation, face_detection, "
    f"facial_expression_recognition (Qwen) and 3d_reconstruction (Llama) — see provenance files.\n"
    f"- face_detection/logo_detection/pose_estimation/visual_reasoning have 1-2 Llama test examples; "
    f"F1 unreliable at that support, excluded from headline claims.\n"
    f"- anomaly_detection has only 7 Llama training examples; systematically confused with "
    f"object_detection — taxonomy-overlap finding, not a bug.\n"]
with open(OUT / "PART4_RESULTS_REPORT.md", "w") as f: f.write("\n".join(report))

verdict = [f"# PART4_SCIENTIFIC_VERDICT.md\n",
    f"All 24 items + full 15-check sanity gate complete, including the cross-teacher "
    f"mixed-training leak fix found during review (see mixed_training_leak_fix_provenance.md).\n",
    f"Cross-teacher drop: Qwen→Llama={distribution_shift_df[distribution_shift_df.training_source=='qwen'].top1_drop.mean():.4f}, "
    f"Llama→Qwen={distribution_shift_df[distribution_shift_df.training_source=='llama'].top1_drop.mean():.4f}\n",
    f"Multi-teacher recovery: {tableD.to_markdown(index=False)}\n",
    f"Human OOD: best={human_agg.loc[human_agg.top1.idxmax()].training_source}-{human_agg.loc[human_agg.top1.idxmax()].architecture} "
    f"(top1={human_agg.top1.max():.4f})\n",
    f"\n**PART4_STATUS = <FILL IN AFTER MANUAL REVIEW>**\n"]
with open(OUT / "PART4_SCIENTIFIC_VERDICT.md", "w") as f: f.write("\n".join(verdict))
print("✅ Both reports written")


# --------------------------------------------------------------------------
# Package
# --------------------------------------------------------------------------
zip_path = "/kaggle/working/KAN_ROUTE_PART4_FINAL"
shutil.make_archive(zip_path, "zip", OUT)
print(f"\n✅ Re-packaged: {zip_path}.zip — download this one before closing the session this time")

In [ ]:
##############################################################################
# Final cleanup — restore provenance files + complete training log, repackage
##############################################################################

with open(OUT / "qwen_corrective_augmentation_provenance.md", "w") as f:
    f.write("# Qwen Corrective Augmentation — Provenance\n\n")
    f.write("**Manually template-authored augmentation, NOT teacher-model-generated.** "
            f"Fixes coverage-starved Shared-28 classes: {STARVED_FIX_CLASSES}. Per-image "
            "phrasing (opener + noun choice) varied so each image's rows form an independent "
            "group under leakage-safe grouping, rather than merging into one atomic block.\n\n")
    f.write("Images reused from existing corpus images already labeled with a person-indicating "
            "tool (person_detection / action_recognition / age_gender_estimation). CLIP image "
            "embeddings pulled from the existing cache; only MiniLM text embeddings freshly "
            "computed. 45 rows added per class across 9 images each.\n\n")
    f.write("**Disclosure requirement:** any Part 4 result touching these classes must cite "
            "this file in Limitations.\n")

with open(OUT / "llama_corrective_augmentation_provenance.md", "w") as f:
    f.write("# Llama Corrective Augmentation — Provenance\n\n")
    f.write(f"**Manually template-authored augmentation, NOT teacher-model-generated.** "
            f"Fixes coverage-starved class: {LLAMA_STARVED_FIX_CLASSES} (raw n=2, both consumed "
            f"by val+test guarantee assignment, leaving zero for train). Same per-image-unique-"
            f"phrasing approach as the Qwen fix. 45 rows added across 9 images, none previously "
            f"used for this class.\n\n")
    f.write("Images reused from existing Llama corpus images (any tool) with a cached CLIP "
            "vector already available; no new encode_image() calls made.\n\n")
    f.write("**Disclosure requirement:** any Part 4 result touching this class must cite this "
            "file in Limitations.\n")
print("✅ Both corrective-augmentation provenance files restored")

# Recover the 12 reused checkpoints' original val accuracies from the previous run's zip,
# so the training log is complete rather than showing only the 6 freshly-trained mixed rows
if PREV_RUN_BASE:
    _prev_log_path = PREV_RUN_BASE / "part4_training_log.csv"
    if _prev_log_path.exists():
        prev_log = pd.read_csv(_prev_log_path)
        prev_qwen_llama = prev_log[prev_log["regime"].isin(["qwen", "llama"])]
        combined_log = pd.concat([prev_qwen_llama, train_log], ignore_index=True)
        combined_log.to_csv(OUT / "part4_training_log.csv", index=False)
        print(f"✅ part4_training_log.csv completed: {len(prev_qwen_llama)} reused + {len(train_log)} fresh = {len(combined_log)} rows")
    else:
        print("⚠️  Previous run's training log not found in PREV_RUN_BASE — "
              "part4_training_log.csv will only show the 6 freshly-trained mixed rows. "
              "Not a correctness issue (checkpoints/results are still valid), just an "
              "incomplete bookkeeping record.")
else:
    print("⚠️  PREV_RUN_BASE unavailable — same caveat as above.")

# Repackage one more time with these restored
zip_path = "/kaggle/working/KAN_ROUTE_PART4_FINAL"
shutil.make_archive(zip_path, "zip", OUT)
print(f"\n✅ Final repackage: {zip_path}.zip — this is the one to download")

In [ ]:
##############################################################################
# Regenerate PART4_SCIENTIFIC_VERDICT.md at full detail (fixes the condensed
# resume version) and re-package. Reads only from OUT's own saved CSVs, so it
# doesn't depend on any in-memory variable surviving from earlier cells.
##############################################################################
import pandas as pd, json, shutil

results = pd.read_csv(OUT / "part4_all_results.csv")
dist_shift = pd.read_csv(OUT / "distribution_shift.csv")
human_agg = pd.read_csv(OUT / "human_ood_summary.csv")
transfer = pd.read_csv(OUT / "shared28_per_class_transfer.csv")
per_class = pd.read_csv(OUT / "part4_per_class_metrics.csv")

agg = results[results.evaluation_source.isin(["qwen_test", "llama_test"])].groupby(
    ["training_source", "architecture", "evaluation_source"])[["top1", "macro_f1"]].mean().reset_index()

def get(ts, arch, ev, col):
    r = agg[(agg.training_source == ts) & (agg.architecture == arch) & (agg.evaluation_source == ev)]
    return r[col].values[0] if len(r) else float("nan")

recovery_rows = []
for arch in ["kan", "mlp"]:
    for target, opp in [("qwen_test", "llama"), ("llama_test", "qwen")]:
        recovery_rows.append(dict(
            Architecture=arch.upper(), Target=target,
            Single_opposite_teacher=get(opp, arch, target, "top1"), Mixed=get("mixed", arch, target, "top1"),
            Recovery_Top1=get("mixed", arch, target, "top1") - get(opp, arch, target, "top1"),
            Recovery_Macro_F1=get("mixed", arch, target, "macro_f1") - get(opp, arch, target, "macro_f1"),
        ))
recoveryD = pd.DataFrame(recovery_rows)

# TRUE in-domain Top-1 — only qwen->qwen_test and llama->llama_test, NOT pooled
# with the cross-teacher rows (that pooling was the bug in the first resume pass)
true_in_domain_top1 = pd.Series([
    get("qwen", "kan", "qwen_test", "top1"), get("qwen", "mlp", "qwen_test", "top1"),
    get("llama", "kan", "llama_test", "top1"), get("llama", "mlp", "llama_test", "top1"),
]).mean()

q_to_l = dist_shift[dist_shift.training_source == "qwen"]
l_to_q = dist_shift[dist_shift.training_source == "llama"]
mixed_recovers = (recoveryD["Recovery_Top1"] > 0).all()
mixed_helps_human = all(
    human_agg[(human_agg.architecture == a) & (human_agg.training_source == "mixed")].top1.values[0] >
    max(human_agg[(human_agg.architecture == a) & (human_agg.training_source == "qwen")].top1.values[0],
        human_agg[(human_agg.architecture == a) & (human_agg.training_source == "llama")].top1.values[0])
    for a in ["kan", "mlp"])
kan_cross = agg[(agg.architecture == "kan") & (agg.evaluation_source.isin(["qwen_test", "llama_test"]))].top1.mean()
mlp_cross = agg[(agg.architecture == "mlp") & (agg.evaluation_source.isin(["qwen_test", "llama_test"]))].top1.mean()
kan_macro_mean = results[results.architecture == "kan"]["macro_f1"].mean()
mlp_macro_mean = results[results.architecture == "mlp"]["macro_f1"].mean()

flagged_classes = ["anomaly_detection", "face_detection", "logo_detection", "pose_estimation", "visual_reasoning"]
def support_row(teacher, cls):
    r = per_class[(per_class.evaluation_source == f"{teacher}_test") & (per_class.training_source == teacher) &
                  (per_class.architecture == "kan") & (per_class.seed == 42) & (per_class.class_name == cls)]
    return int(r["support"].values[0]) if len(r) else None

support_table_md = "| Class | Llama test support | Qwen test support |\n|---|---|---|\n"
for cls in flagged_classes:
    support_table_md += f"| {cls} | {support_row('llama', cls)} | {support_row('qwen', cls)} |\n"

anomaly_pred = pd.read_csv(PRED_DIR / "kan_llama_seed42_on_llama_test.csv")
anomaly_confusion_text = "Prediction file missing enriched columns."
if "ground_truth_tool" in anomaly_pred.columns and "pred_top1_tool" in anomaly_pred.columns:
    subset = anomaly_pred[anomaly_pred["ground_truth_tool"] == "anomaly_detection"]
    counts = subset["pred_top1_tool"].value_counts()
    anomaly_confusion_text = "\n".join(f"  {k}: {v}" for k, v in counts.items())

verdict = []
verdict.append("# PART4_SCIENTIFIC_VERDICT.md\n")
verdict.append(f"1. **Substantial teacher-specific distribution shift?** Mean Top-1 drop: "
    f"Qwen→Llama={q_to_l.top1_drop.mean():.4f}, Llama→Qwen={l_to_q.top1_drop.mean():.4f}. "
    f"{'Yes — both drops exceed 35pp.' if max(q_to_l.top1_drop.mean(), l_to_q.top1_drop.mean()) > 0.35 else 'Modest.'}\n")
verdict.append(f"2. **Qwen→Llama degradation:** {q_to_l.top1_drop.mean():.4f} Top-1, {q_to_l.macro_f1_drop.mean():.4f} Macro-F1.\n")
verdict.append(f"3. **Llama→Qwen degradation:** {l_to_q.top1_drop.mean():.4f} Top-1, {l_to_q.macro_f1_drop.mean():.4f} Macro-F1.\n")
verdict.append(f"4. **Does multi-teacher training recover performance?** "
    f"{'Yes, for every architecture/target combination' if mixed_recovers else 'Mixed — check table'} "
    f"(recovery ranges {recoveryD.Recovery_Top1.min():.4f}-{recoveryD.Recovery_Top1.max():.4f} Top-1).\n")
verdict.append(f"5. **Does multi-teacher training improve human-query generalization?** "
    f"{'Yes, for both KAN and MLP' if mixed_helps_human else 'Not for every architecture'}. "
    f"Llama-only generalizes to human queries substantially worse than Qwen-only "
    f"({human_agg[human_agg.training_source=='llama'].top1.mean():.4f} vs "
    f"{human_agg[human_agg.training_source=='qwen'].top1.mean():.4f} mean Top-1).\n")
verdict.append(f"6. **Is KAN more robust than MLP under distribution shift?** Pooled cross/in-domain "
    f"Top-1 — KAN={kan_cross:.4f}, MLP={mlp_cross:.4f}. Pooled Macro-F1 — KAN={kan_macro_mean:.4f}, MLP={mlp_macro_mean:.4f}.\n")
verdict.append(f"7. **Is KAN's advantage Top-1, Macro-F1, or interpretability?** "
    f"Top-1 gap ({kan_cross-mlp_cross:+.4f}) vs. Macro-F1 gap ({kan_macro_mean-mlp_macro_mean:+.4f}) "
    f"— {'concentrated in Macro-F1' if abs(kan_macro_mean-mlp_macro_mean) > abs(kan_cross-mlp_cross) else 'roughly proportional across both'}, not raw Top-1 alone.\n")
gap_pp = (true_in_domain_top1 - human_agg.top1.max()) * 100
verdict.append(f"8. **Are the human results strong enough for a Q1-journal claim?** Best human "
    f"canonical Top-1 = {human_agg.top1.max():.4f} ({human_agg.loc[human_agg.top1.idxmax(),'training_source']}-"
    f"{human_agg.loc[human_agg.top1.idxmax(),'architecture']}) vs. TRUE in-domain synthetic Top-1 "
    f"≈ {true_in_domain_top1:.4f} (qwen→qwen_test + llama→llama_test only, NOT pooled with "
    f"cross-teacher rows) — roughly a {gap_pp:.0f}pp synthetic→human gap. Present this as the "
    f"honest synthetic-to-real transfer finding, not a routing-solved claim.\n")
verdict.append("## 9. Per-class anomaly investigation\n")
verdict.append(f"Test-set support for flagged classes:\n\n{support_table_md}\n")
verdict.append(f"`anomaly_detection` predictions on its own Llama in-domain test set:\n\n{anomaly_confusion_text}\n\n"
    "If every prediction lands on one other class, that's a taxonomy-overlap finding, not a bug. "
    "Classes with test support ≤2 should be excluded from headline per-class claims — F1 at n≤2 "
    "can only be 0.0 or 1.0.\n")
verdict.append("10. **Is another experiment scientifically necessary?** Review items 1-9 against "
    "your own judgment before deciding.\n")
verdict.append("\n**PART4_STATUS = <FILL IN AFTER YOUR OWN REVIEW>**\n")
verdict.append("\n> All numbers above recomputed directly from this session's OUT CSVs and raw "
    "predictions — not carried over from any prior run's text.\n")

with open(OUT / "PART4_SCIENTIFIC_VERDICT.md", "w") as f:
    f.write("\n".join(verdict))

print(f"True in-domain Top-1: {true_in_domain_top1:.4f}")
print(f"Best human Top-1: {human_agg.top1.max():.4f}")
print(f"Gap: {gap_pp:.1f}pp")
print("✅ PART4_SCIENTIFIC_VERDICT.md regenerated at full detail")

# Re-package
zip_path = "/kaggle/working/KAN_ROUTE_PART4_FINAL"
shutil.make_archive(zip_path, "zip", OUT)
print(f"✅ Re-packaged: {zip_path}.zip — download this one")